# Python Lambda

> 📘 **Python Mastery** · Module 03 — Functions · Lesson 5/5

A lambda is a tiny anonymous function written in one line. You rarely call lambdas directly — you *hand them to* other functions such as `sorted`, `map`, and `filter`, telling those functions exactly how to behave for a moment. Final lesson of the module: small syntax, huge leverage.

## 🎯 Learning Objectives

By the end of this lesson you will be able to:

- **Write** lambda expressions and predict their results
- **Translate** between a lambda and its equivalent `def`
- **Sort** anything with `sorted(..., key=lambda ...)`
- **Transform and filter** sequences with `map()` and `filter()` — remembering they return iterators
- **Use** lambdas inside `min`, `max`, `.sort()` and multi-parameter helpers
- **Respect** the limitations and the PEP 8 guidance on naming lambdas

## 1. Lambda: an Anonymous One-Line Function

`lambda` creates a function **without a name** in a single expression: parameters, a colon, then exactly one expression — whose value is automatically returned. No `def`, no `return`, no statements.

Perfect for the many moments when you need a function for one instant and will never need it again.

**Syntax:**

```python
lambda parameters: expression

square = lambda n: n * n     # usually assigned briefly, or passed inline
square(5)                    # -> 25
```

**Example:**

In [1]:
square = lambda n: n * n
print(square(5))

add = lambda a, b: a + b
print(add(3, 4))

to_upper = lambda word: word.upper()
print(to_upper("dhaka"))

# No parameters at all -- the expression's VALUE is the result
celebrate = lambda: "Victory day!"
print(celebrate())

25
7
DHAKA
Victory day!


## 2. lambda vs def: Same Machinery

Every lambda has a `def` twin, and both produce the very same kind of object: a function. The differences are surface-level — `def` requires a name and an indented block; a lambda is an *expression*, so it fits inside other expressions (like a function argument).

> 🔍 **Under the Hood:** both compile to essentially identical code objects. One giveaway: an unnamed lambda's `__name__` is literally `"<lambda>"` — which is also why tracebacks involving lambdas are so unhelpful.

**Syntax:**

```python
# these two behave identically
def square(n):
    return n * n

square = lambda n: n * n
```

**Example:**

In [2]:
def square_def(n):
    return n * n

square_lambda = lambda n: n * n


print(square_def(7), square_lambda(7))
print(type(square_def).__name__, type(square_lambda).__name__)
print(square_def.__name__, "->", square_lambda.__name__)

# Identical behaviour across a range of inputs:
print(all(square_def(i) == square_lambda(i) for i in range(-3, 4)))

49 49
function function
square_def -> <lambda>
True


## 3. Why Lambdas Exist: Throwaway Helpers

Naming a function is a small act of design: choose a place, choose a good name, maybe write a docstring. For a function used exactly once, that ceremony outweighs the benefit — so Python lets you define the behaviour **right where it is used**.

Rule of thumb: if the function deserves a sentence of explanation, give it a `def` and a name; if it is obvious at a glance, a lambda is fine.

**Syntax:**

```python
sorted(data, key=lambda item: item[1])    # helper used once, defined inline
```

**Example:**

In [3]:
students = [("Sarah", 92), ("Rahim", 85), ("Aisha", 95)]

# Option A: a named helper -- great when it is reused or non-trivial
def by_score(pair):
    return pair[1]

print(sorted(students, key=by_score))

# Option B: the same idea inline -- nobody ever needs 'by_score' afterwards
print(sorted(students, key=lambda pair: pair[1]))

[('Rahim', 85), ('Sarah', 92), ('Aisha', 95)]
[('Rahim', 85), ('Sarah', 92), ('Aisha', 95)]


## 4. Sorting with `key=lambda`

By default `sorted()` compares items directly. Supply `key=` — a function applied to every item — and Python instead orders the **results**. This is the lambda's natural habitat: "sort products by price", "sort words by length".

Add `reverse=True` for descending order. `sorted()` leaves the original untouched; `list.sort()` reorders in place.

**Syntax:**

```python
sorted(iterable, key=lambda item: sort_value, reverse=False)
list.sort(key=lambda item: sort_value)      # in-place variant
```

**Example:**

In [4]:
products = [
    ("Notebook", 120),
    ("Pen", 25),
    ("Backpack", 1450),
    ("Calculator", 640),
]

# Sort tuples by price (their second element)
by_price = sorted(products, key=lambda item: item[1])
print(by_price)

# Descending: most expensive first
print(sorted(products, key=lambda item: item[1], reverse=True))

words = ["banana", "fig", "cherry", "date"]
print(sorted(words, key=len))                  # built-in len works as key too
print(sorted(words, key=lambda w: w[-1]))      # by LAST letter

[('Pen', 25), ('Notebook', 120), ('Calculator', 640), ('Backpack', 1450)]
[('Backpack', 1450), ('Calculator', 640), ('Notebook', 120), ('Pen', 25)]
['fig', 'date', 'banana', 'cherry']
['banana', 'date', 'fig', 'cherry']


## 5. `key` in `min`, `max`, and `.sort()`

The `key` idea is universal across Python's ordering tools: `min`, `max`, `list.sort`, and beyond. Wherever Python ranks things, it accepts the same `key=` function — learn it once, use it everywhere. With `min`/`max` the key decides which item is smallest/largest, not what is returned: you get back the **original item**.

**Syntax:**

```python
min(items, key=lambda item: sort_value)     # item WITH the smallest sort_value
max(items, key=lambda item: sort_value)     # item WITH the largest sort_value
items.sort(key=lambda item: sort_value)     # sorts the list in place
```

**Example:**

In [5]:
products = [
    ("Notebook", 120),
    ("Pen", 25),
    ("Backpack", 1450),
]

cheapest = min(products, key=lambda p: p[1])
priciest = max(products, key=lambda p: p[1])
print(cheapest, "|", priciest)

players = [("Sarah", 92), ("Rahim", 85), ("Aisha", 95)]
top = max(players, key=lambda p: p[1])
print("Top scorer:", top[0])

# In-place variant on a plain list:
inventory = ["mango", "apple", "blueberry"]
inventory.sort(key=len)
print(inventory)

('Pen', 25) | ('Backpack', 1450)
Top scorer: Aisha
['mango', 'apple', 'blueberry']


## 6. `map()` and `filter()`

Two classic partners for lambdas:

- `map(function, iterable)` applies the function to **every** item — a transformation.
- `filter(function, iterable)` keeps items for which the function returns truthy — a selection.

Both return **lazy iterators**, not lists: nothing is produced until you ask, and once consumed they are empty. Wrap them in `list(...)` to see their contents.

> 🔍 **Under the Hood:** laziness means no work happens until an element is requested. Chain several `map`s over a million-row dataset and memory stays flat — each row flows through the pipeline one at a time instead of building a million-element intermediate list per step.

**Syntax:**

```python
iterator = map(lambda x: ..., iterable)      # transform every element
iterator = filter(lambda x: ..., iterable)   # keep elements passing the test
values = list(iterator)                       # materialise ONCE, then reuse
```

**Example:**

In [6]:
prices_taka = [120, 250, 80]

# map: transform every element
with_vat = map(lambda p: p * 1.15, prices_taka)
print(with_vat)                        # an iterator OBJECT, not the values
print(list(with_vat))                  # materialise with list()

# filter: keep elements passing the test
temps = [36.2, 38.9, 37.0, 40.1, 35.8]
feverish = filter(lambda t: t >= 38.0, temps)
print(list(feverish))

# Chain them: transform only the survivors
scores = [45, 82, 67, 91, 58]
curved = map(lambda s: s + 5, filter(lambda s: s >= 60, scores))
print(list(curved))

[138.0, 287.5, 92.0]
[38.9, 40.1]
[87, 72, 96]


In [7]:
numbers = [1, 2, 3, 4]
doubled = map(lambda n: n * 2, numbers)

print(list(doubled))     # [2, 4, 6, 8]  -- consumed HERE
print(list(doubled))     # []             -- iterators never rewind!

# List comprehensions: the idiomatic alternative for scripts
print([n * 2 for n in numbers])
print([n for n in numbers if n % 2 == 0])

[2, 4, 6, 8]
[]
[2, 4, 6, 8]
[2, 4]


## 7. Multi-Argument Lambdas

Before the colon you may list any parameters — several of them, with defaults, even `*args`. What you cannot do is use more than one *statement*: the single-expression rule stands.

Multi-parameter lambdas shine wherever a binary operation is expected, such as `functools.reduce`, which folds a sequence into a single value two elements at a time.

**Syntax:**

```python
lambda a, b: a + b                    # two parameters
lambda name, punctuation="!": ...     # default allowed
combine(*args)                        # even *args allowed
```

**Example:**

In [8]:
power = lambda base, exponent: base ** exponent
print(power(2, 10))

greet = lambda name, punctuation="!": f"Hello {name}{punctuation}"
print(greet("Sarah"), greet("Rahim", "."))

flex = lambda *args: sum(args) / len(args)
print(flex(80, 90, 100))

from functools import reduce

total = reduce(lambda acc, price: acc + price, [120, 250, 80], 0)
print(total)

1024
Hello Sarah! Hello Rahim.
90.0
450


## 8. Limitations and PEP 8 Guidance

Lambdas deliberately lack features:

- **No statements** — no assignments, no loops, no `return` keyword, no multiline logic.
- **No annotations** — type hints inside a lambda are a syntax error.
- **No useful name** — tracebacks only say `<lambda>`.

PEP 8 adds a style warning: **do not bind a lambda to a name** (`f = lambda x: ...`). If a function deserves a name, it deserves `def` — with a docstring and readable tracebacks. Save lambdas for inline duty: `key=`, `map()`, `filter()`.

❌

```python
f = lambda x: 2 * x        # PEP 8 violation: named lambda
```

✅

```python
def double(x):
    """Return twice the input."""
    return 2 * x
```

**Example:** prove the limits safely with the parser.

In [9]:
bad_snippets = [
    "f = lambda x: y = x + 1",     # assignment is a STATEMENT
    "g = lambda x: import os",     # import is a STATEMENT
]
for snippet in bad_snippets:
    try:
        compile(snippet, "<demo>", "exec")
    except SyntaxError as err:
        print(f"SyntaxError in {snippet!r}: {err.msg}")

# Annotations are unsupported too:
try:
    compile("h = lambda x: int: x", "<demo>", "exec")
except SyntaxError as err:
    print("SyntaxError:", err.msg)

SyntaxError in 'f = lambda x: y = x + 1': cannot assign to lambda
SyntaxError in 'g = lambda x: import os': invalid syntax
SyntaxError: invalid syntax


## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
| --- | --- | --- |
| Assigning a lambda to a name (`f = lambda ...`) | PEP 8 violation; tracebacks show only `<lambda>` | Use `def` with a real name and docstring |
| Squeezing statements or multi-line logic into a lambda | `SyntaxError` — one expression only | Graduate to `def` with a proper body |
| Consuming a `map`/`filter` iterator twice | Second consumption yields nothing — iterators are one-shot | Materialise once: `data = list(iterator)` |
| Sorting tuples/dicts without `key=` | Tuples compare lexicographically; dicts raise `TypeError` | Always pass `key=lambda item: ...` explicitly |
| Writing clever nested lambdas | Unreadable one-liners nobody can debug | Expand into named functions — clarity wins |

## 💡 Best Practices & Pro Tips

- Keep lambda bodies trivially readable at a glance (`item[1]`, `len(w)`, `p.price`). Anything longer becomes a `def`.
- Reach for `key=lambda ...` constantly — sorting, `min`, `max` — it is the highest-value use of lambdas in real code.
- In everyday scripts, prefer list comprehensions over `map`/`filter` for readability; keep `map`/`filter` for streaming-style pipelines where laziness matters.
- Remember iterator semantics: lazy **and** one-shot. Wrap in `list()` if you need the values more than once.
- 🤖 **AI-engineering relevance:** quick experiments define activations and scoring tricks inline (`lambda z: 1 / (1 + np.exp(-z))`). Data pipelines chain lazy transforms exactly like a `torch` DataLoader, and pandas' `df.sort_values(by=...)` is `key=`-based sorting wearing a friendlier coat. Fluent `key=` thinking transfers directly.

## 📌 Summary

| Tool | What it does | Example |
| --- | --- | --- |
| `lambda params: expr` | Anonymous single-expression function | `lambda n: n * n` |
| `sorted(xs, key=f)` | Order by each item's key value | `sorted(students, key=lambda p: p[1])` |
| `xs.sort(key=f)` | Same, in place | `inventory.sort(key=len)` |
| `min(xs, key=f)` / `max(xs, key=f)` | Item with smallest/largest key | `min(products, key=lambda p: p[1])` |
| `map(f, xs)` | Lazy transform of every element | `map(lambda p: p * 1.15, prices)` |
| `filter(f, xs)` | Lazy selection of passing elements | `filter(lambda t: t >= 38, temps)` |
| `list(iterator)` | Materialise a lazy iterator once | `list(map(...))` |
| `reduce(f, xs, start)` | Fold sequence into one value | `reduce(lambda a, b: a + b, xs, 0)` |

Key takeaways:

- A lambda is a `def` compressed to one expression — same machinery, no name.
- Lambdas exist to be handed over instantly: `key=`, `map()`, `filter()`.
- `map`/`filter` return lazy, one-shot iterators — wrap in `list()` to inspect.
- Named logic deserves `def`; inline throwaway logic earns its lambda.

## 🔗 Next Lesson

🎉 **Module 03 complete!** Continue with **04_Data_Structures** — lists, tuples, dictionaries and sets: the containers your new functions will chew on.